# Deep SRQ PATH Restart Ablation

Compares PATH start strategies for DeepSRQ SRE solves: pure starts plus random restarts, pure starts only, and random starts only.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "bimatrix_game":
    BIMATRIX_DIR = ROOT
else:
    BIMATRIX_DIR = ROOT / "discrete_action_space" / "bimatrix_game"
DISCRETE_DIR = BIMATRIX_DIR.parent
for path in (BIMATRIX_DIR, DISCRETE_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from experiment_harness import (
    BASE_SEED,
    configure_path_runtime,
    run_deep_srq_ablation_variants,
    summarize_ablation_timing_rows,
)
from stats_utils import save_training_stats

In [ ]:
PATHWRAP = str(configure_path_runtime(DISCRETE_DIR))
OUTPUT_ROOT = BIMATRIX_DIR / "ablation_runs" / "path_restarts"

SCENARIOS = ("scenario1", "scenario3")
N_EPISODES = 3000
EPSILON = 0.5
EPSILON_SCHEDULE = "linear"
USE_GPU = True
SOLVER_NAME = "path_c"

RUN_VARIANTS = (
    {
        "label": "pure_plus_random20",
        "hyperparameter_overrides": {"sre_num_repeats": 20, "sre_include_pure_starts": True},
    },
    {
        "label": "pure_plus_random10",
        "hyperparameter_overrides": {"sre_num_repeats": 10, "sre_include_pure_starts": True},
    },
    {
        "label": "pure_only",
        "hyperparameter_overrides": {"sre_num_repeats": 0, "sre_include_pure_starts": True},
    },
    {
        "label": "random10_only",
        "hyperparameter_overrides": {"sre_num_repeats": 10, "sre_include_pure_starts": False},
    },
    {
        "label": "random5_only",
        "hyperparameter_overrides": {"sre_num_repeats": 5, "sre_include_pure_starts": False},
    },
)

In [ ]:
results = run_deep_srq_ablation_variants(
    variants=RUN_VARIANTS,
    scenarios=SCENARIOS,
    base_seed=BASE_SEED,
    pathwrap_path=PATHWRAP,
    output_root=OUTPUT_ROOT,
    use_gpu=USE_GPU,
    write_plots=False,
    default_n_episodes=N_EPISODES,
    default_epsilon_robust_initial=EPSILON,
    default_epsilon_schedule=EPSILON_SCHEDULE,
    default_solver_name=SOLVER_NAME,
)

save_training_stats(OUTPUT_ROOT / "path_restarts_manifest.txt", results)

In [ ]:
for row in summarize_ablation_timing_rows(results):
    print(row)